In [17]:
# ----------------------------------------------------
# 1. CORE LANGCHAIN RETRIEVAL & PROCESSING
# ----------------------------------------------------
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_chroma import Chroma
#from langchain_huggingface import HuggingFaceEmbeddings
from langchain_litellm import LiteLLMEmbeddings
from config import Config
from model import llm, chat_template, get_ai_response, get_ai_response_modern
# ----------------------------------------------------
# 2. PROMPTS, MEMORY & CHAIN CONSTRUCTS
# ----------------------------------------------------
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

#from langchain.chains import RetrievalQA, ConversationalRetrievalChain
#from langchain.chains import create_retrieval_chain
#from langchain.chains.combine_documents import create_stuff_documents_chain
#from langchain.memory import ConversationBufferMemory
from langchain_classic.chains import create_retrieval_chain, create_history_aware_retriever
from langchain_classic.chains.combine_documents import create_stuff_documents_chain





# ----------------------------------------------------
# 3. LITELLM REPLACEMENT (Replaces all ibm_watsonx imports)
# ----------------------------------------------------
from langchain_litellm import ChatLiteLLM
from config import Config # Imports your dynamic .env credentials config setup

# (Optional utility from your course)
import wget



In [7]:
filename = 'companyPolicies.txt'
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/6JDbUb_L3egv_eOkouY71A.txt'

# Use wget to download the file
wget.download(url, out=filename)
print('file downloaded')

100% [..............................................................................] 15660 / 15660file downloaded


In [8]:
with open(filename, 'r') as file:
    # Read the contents of the file
    contents = file.read()
    print(contents)

1.	Code of Conduct

Our Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.
Integrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.
Respect: We embrace diversity and value each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. We create an inclusive environment where differences are celebrated and everyone is treated with dignity and courtesy.
Accountability: We take responsibility for our actions and decisions. We follow all relevant laws and regulations, and we strive to continuously improve our practices. We report any potential violations of 

In [9]:
loader = TextLoader(filename)
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)
print(len(texts))

Created a chunk of size 1624, which is longer than the specified 1000
Created a chunk of size 1885, which is longer than the specified 1000
Created a chunk of size 1903, which is longer than the specified 1000
Created a chunk of size 1729, which is longer than the specified 1000
Created a chunk of size 1678, which is longer than the specified 1000
Created a chunk of size 2032, which is longer than the specified 1000
Created a chunk of size 1894, which is longer than the specified 1000


16


In [13]:
#embeddings = HuggingFaceEmbeddings()
#docsearch = Chroma.from_documents(texts, embeddings)  # store the embedding in docsearch using Chromadb
#print('document ingested')
# instead of processing heavy computation and storage locally with pytorch and hugging face using litellm API

# 1. Initialize cloud-based embeddings using LiteLLM
# This instantly routes via API instead of compiling PyTorch locally
embedding_model = "openai/text-embedding-3-small"
embeddings = LiteLLMEmbeddings(model=embedding_model)

# 2. Store the embedding structures in Chromadb natively
docsearch = Chroma.from_documents(texts, embeddings)
print('🎉 Document ingested successfully via LiteLLM API!')

🎉 Document ingested successfully via LiteLLM API!


In [16]:
model_id = llm

In [29]:

# 1. Setup a standard RAG prompt template framework
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, say that you don't know.\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
   # ("human", "Context:\n{context}\n\nQuestion: {input}") Comment above two lines and uncomment this to see vague answers without instrutions
])

# 2. Build the Document Combination Node
# This replaces the old 'chain_type="stuff"' parameter
question_answer_chain = create_stuff_documents_chain(llm, prompt)

# 3. Assemble the final compilation pipeline
# This replaces the old 'RetrievalQA.from_chain_type' method
qa = create_retrieval_chain(docsearch.as_retriever(), question_answer_chain)


In [24]:
# Define your targeted search prompt
query = "what is mobile policy?"

# Execute the modern LCEL compilation pipeline
result = qa.invoke({"input": query})

# Print out your clean string response answer node
print("Answer:", result["answer"])


Answer: # Mobile Phone Policy

The Mobile Phone Policy is a set of standards and expectations that govern how employees should use mobile devices within the organization. Its main purpose is to ensure responsible usage that aligns with company values and legal compliance.

## Key Components:

1. **Acceptable Use**: Mobile devices are mainly for work-related tasks, with limited personal use allowed as long as it doesn't interfere with work responsibilities.

2. **Security**: Employees must protect their devices and login credentials, be careful when downloading apps or clicking unknown links, and report any security concerns immediately.

3. **Confidentiality**: Employees should not send sensitive company information through unsecured messaging apps or emails, and must be discreet when discussing company matters in public.

4. **Cost Management**: Personal and company phone usage should be kept separate, with employees reimbursing any personal charges made on company-issued phones.

5. 

In [20]:
# Asking a higher level question prompt
query = "Can you summarize the document for me?"

# Execute the modern LCEL compilation pipeline
result = qa.invoke({"input": query})

# Print out your clean string response answer node
print("Answer:", result["answer"])




Answer: Based on the provided context, here's a summary of the key policies:

## Document Summary

**1. Health and Safety Policy**
- Prioritizes the well-being of employees, customers, and the public
- Commits to compliance with all health and safety laws
- Aims to maintain a hazard-free workplace
- Emphasizes that safety is everyone's responsibility
- Includes regular safety assessments, training, and open communication about concerns

**2. Anti-Discrimination and Harassment Policy**
- Prohibits discrimination based on race, color, religion, gender, national origin, age, disability, sexual orientation, or any protected characteristic
- Zero tolerance for harassment in any form
- Covers all aspects of employment (recruitment, hiring, compensation, promotions, etc.)
- Encourages prompt reporting of incidents with confidential investigation procedures
- Violations may result in disciplinary action, including termination

**3. Discipline and Termination Policy**
- Sets clear performance a

In [30]:
# Try Asking a a query which it may not have an answer to
query = "Can I eat in company vehicles?"

# Execute the modern LCEL compilation pipeline
result = qa.invoke({"input": query})

# Print out your clean string response answer node
print("Answer:", result["answer"])


Answer: Based on the provided context, I don't know the answer to that question. The documents provided include policies on smoking, drugs and alcohol, and mobile phone usage, but there is no information about eating in company vehicles.

The smoking policy does state that "Smoking is not permitted in company vehicles, whether they are owned or leased, to maintain the condition and cleanliness of these vehicles," but this only addresses smoking, not eating.


In [31]:
#  Asking a query trying to check memory
query = "what can I not do in it?"

# Execute the modern LCEL compilation pipeline
result = qa.invoke({"input": query})

# Print out your clean string response answer node
print("Answer:", result["answer"])


Answer: Based on the policies provided, here are the key things you **cannot do**:

## Internet and Email Policy - Prohibited Actions:
- Share passwords or login credentials
- Use internet/email for harassment, discrimination, or distributing offensive/inappropriate content
- Transmit confidential information, trade secrets, or sensitive customer data via email without encryption
- Violate copyright laws or data protection regulations

## Smoking Policy - Prohibited Actions:
- Smoke inside company buildings, offices, meeting rooms, or any enclosed spaces
- Smoke outside designated smoking areas
- Use electronic cigarettes or vaping devices in prohibited areas
- Smoke in company vehicles (owned or leased)
- Litter cigarette butts or smoking materials on company premises

## Mobile Phone Policy - Prohibited Actions:
- Allow personal mobile use to disrupt work obligations
- Transmit sensitive company information via unsecured messaging apps or emails
- Fail to report lost or stolen mobile

In [33]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.messages import HumanMessage, AIMessage

# 1. Setup a standard RAG prompt template framework
# We include MessagesPlaceholder so the chain knows where to inject past messages
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, say that you don't know.\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder(variable_name="chat_history"), # Handles past memory
    ("human", "{input}"),                              # Handles new question
])

# 2. Build the Document Combination Node
# This handles the {context} formatting automatically under the hood!
qa_chain = create_stuff_documents_chain(llm, prompt)

# 3. Initialize an empty list to track conversation memory
chat_history = []


In [34]:
query_1 = "what is mobile policy?"

# Execute the chain
# You must pass the retrieved documents manually to 'context' using .invoke()
retrieved_docs = docsearch.as_retriever().invoke(query_1)

response_1 = qa_chain.invoke({
    "input": query_1,
    "context": retrieved_docs,
    "chat_history": chat_history
})

print("Answer 1:", response_1)

# Save Turn 1 to your memory list
chat_history.extend([
    HumanMessage(content=query_1),
    AIMessage(content=response_1)
])


Answer 1: Based on the provided context, the **Mobile Phone Policy** is a set of standards and expectations that govern the appropriate and responsible use of mobile devices in the organization. Here are the key points:

**Main Purpose:**
- Ensure employees use mobile phones in a manner consistent with company values and legal compliance

**Key Guidelines:**

1. **Acceptable Use**: Mobile devices are primarily for work-related tasks, with limited personal usage allowed as long as it doesn't disrupt work

2. **Security**: Employees must safeguard devices and credentials, be cautious with downloads and links, and report security concerns promptly

3. **Confidentiality**: Avoid transmitting sensitive company information through unsecured messaging or emails; be discreet in public spaces

4. **Cost Management**: Keep personal usage separate from company accounts and reimburse any personal charges on company-issued phones

5. **Compliance**: Follow all relevant laws and regulations regardin

In [35]:
query_2 = "Does it apply to tablets too?"

# Retrieve context based on the new query
retrieved_docs_2 = docsearch.as_retriever().invoke(query_2)

response_2 = qa_chain.invoke({
    "input": query_2,
    "context": retrieved_docs_2,
    "chat_history": chat_history # Contains past turn history!
})

print("Answer 2:", response_2)

# Save Turn 2 to your memory list
chat_history.extend([
    HumanMessage(content=query_2),
    AIMessage(content=response_2)
])


Answer 2: Based on the provided context, the policy refers to "**mobile devices**" in several places, not just mobile phones specifically. 

For example, the context states:
- "The Mobile Phone Policy sets forth the standards and expectations governing the appropriate and responsible usage of **mobile devices**"
- "Mobile **devices** are primarily intended for work-related tasks"
- "Safeguard your mobile **device** and access credentials"

This broader language of "mobile devices" would typically include tablets as well as smartphones. However, the policy document provided doesn't explicitly list tablets or specify exactly which types of devices are covered beyond "mobile devices."

For absolute clarity on whether tablets are included, you may want to check with your IT department or HR, as they can confirm the specific devices covered under this policy.


In [ ]:
def qa_modern():
    # 1. Setup prompt to contextualize history (reformulates questions so search queries stay clean)
    contextualize_q_system_prompt = (
        "Given a chat history and the latest user question "
        "which might reference context in the chat history, "
        "formulate a standalone question which can be understood "
        "without the chat history. Do NOT answer the question, "
        "just reformulate it if needed and otherwise return it as is."
    )
    contextualize_q_prompt = ChatPromptTemplate.from_messages([
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ])
    
    # 2. Compile the history-aware retrieval sub-chain
    history_aware_retriever = create_history_aware_retriever(
        llm=llm, 
        retriever=docsearch.as_retriever(), 
        prompt=contextualize_q_prompt
    )

    # 3. Setup the final core QA prompt framework
    system_prompt = (
        "You are an assistant for question-answering tasks. "
        "Use the following pieces of retrieved context to answer "
        "the question. If you don't know the answer, say that you "
        "don't know.\n\n"
        "Context:\n{context}"
    )
    qa_prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ])
    
    # 4. Assemble the unified compilation pipeline
    question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
    rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

    # 5. Run the session loop managing states via rich Message objects
    chat_history = []
    print("🚀 Modern Chatbot initialized. Type 'exit' to quit.")
    
    while True:
        query = input("Question: ")
        
        if query.lower() in ["quit", "exit", "bye"]:
            print("Answer: Goodbye!")
            break
            
        # Execute invocation safely via clean key mappings
        response = rag_chain.invoke({"input": query, "chat_history": chat_history})
        answer = response["answer"]
        
        # Append rich structural history tags
        chat_history.extend([
            HumanMessage(content=query),
            AIMessage(content=answer)
        ])
        
        print("Answer: ", answer)
